# Churn model

Here we build the model that predicts who is more likely to leave, train it, and see how it performs.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from preparar_datos import load, clean, to_numeric, prepare_for_model

# Load from data/ (one folder up from notebooks/)
csv_path = Path.cwd().parent / "data"
csv_files = list(csv_path.glob("*.csv"))
df = load(csv_files[0]) if csv_files else load()
df = clean(df)
df = to_numeric(df)
X, y = prepare_for_model(df)
X.head()

Split into a part to train on and a part to test on (so we're not cheating and we see how it really does).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train a simple baseline and two real models so we can compare. We care most about catching churners (the minority class).

In [ ]:
# Baseline: guess at random following the same proportion of churn we have in train
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)

# Simple model that often does okay and is easy to explain
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train, y_train)

In [ ]:
import xgboost as xgb

pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
model = xgb.XGBClassifier(scale_pos_weight=pos_weight, random_state=42, use_label_encoder=False, eval_metric="logloss")
model.fit(X_train, y_train)

Compare all three: see which one does best. We care especially about recall for churn (catching leavers).

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
import pandas as pd

results = []
for name, m in [("Baseline (dummy)", dummy), ("Logistic Regression", logreg), ("XGBoost", model)]:
    pred = m.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "F1": round(f1_score(y_test, pred), 4),
        "Recall (churn)": round(recall_score(y_test, pred, pos_label=1), 4),
    })
comparison = pd.DataFrame(results)
comparison

# Save so we can show results in the repo after running
reports_path = Path.cwd().parent / "reports"
reports_path.mkdir(exist_ok=True)
comparison.to_csv(reports_path / "model_comparison.csv", index=False)
print("Saved to reports/model_comparison.csv")

See how it did: how many it got right and wrong (we especially care about not missing churners).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.xlabel("What the model said")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.show()

Which variables matter most for the model saying "leave" or "stay" (so we understand what it's looking at).

In [ ]:
import shap

explainer = shap.TreeExplainer(model, X_train)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, show=False)
plt.title("How much each variable influences the prediction")
plt.tight_layout()
reports_path = Path.cwd().parent / "reports"
reports_path.mkdir(exist_ok=True)
plt.savefig(reports_path / "shap_summary.png", dpi=150)
plt.show()

Save the model so we can use it later without training again.

In [ ]:
import pickle

models_path = Path.cwd().parent / "models"
models_path.mkdir(exist_ok=True)
with open(models_path / "modelo_churn.pkl", "wb") as f:
    pickle.dump(model, f)
with open(models_path / "columnas.pkl", "wb") as f:
    pickle.dump(list(X.columns), f)
print("Done, model saved to models/")